# Prüfung der Datenbanken und Rohdaten

**Datenbanken** (alte gegen neue Datenbank, Integrität der neuen)

| Teil | Inhalt |
|---|---|
| A | Schema und Signalinventar |
| B | Integrität der neuen Datenbank |
| C | Duplikate: Herkunft, Zeitauflösung |
| D | Metadaten-Fanout |
| E | WCS_Y_mm |
| F | Zugriffsleistung |


Die beiden Datenbanken enthalten **verschiedene Kampagnen** – Wertebereiche pro Signal sind deshalb nicht vergleichbar, nur Schema, Signalinventar und Anteile der Ströme.

## Datenbanken

### Konfiguration

In [ ]:
import json
import os
import tempfile
import time
from pathlib import Path

import duckdb
import pandas as pd

# Pfade relativ zum Notebook-Verzeichnis
HERE = Path.cwd()
DB_NEW = HERE / "../databases/DBnew.duckdb"
DB_OLD = HERE / "../databases/DBold.duckdb"
TBL_NEW = "cleaned_data"
TBL_OLD = "my_table"

HF_PERIOD_MS = 2.0
PROBE_SIGNAL = "TORQUE|3"
SUSPECT_SIGNAL = "actFeedRateIpo[u1]"
OUT_CSV = HERE / "messung_kapitel7.csv"

PLATTE = 40
NUT = 1

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)

### Hilfsfunktionen

In [ ]:
def header(text: str) -> None:
    print(f"\n{'=' * 78}\n{text}\n{'=' * 78}")


def sub(text: str) -> None:
    print(f"\n--- {text} ---")


def show(df: pd.DataFrame, empty: str = "keine") -> None:
    print(df.to_string(index=False) if not df.empty else empty)


def lit(value) -> str:
    return f"'{value}'" if isinstance(value, str) else str(value)


def connect() -> duckdb.DuckDBPyConnection:
    """Neue DB als Hauptdatenbank, alte DB schreibgeschuetzt daneben."""
    for p in (DB_NEW, DB_OLD):
        if not p.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {p}")
    con = duckdb.connect(str(DB_NEW), read_only=True)
    con.execute(f"ATTACH '{DB_OLD.as_posix()}' AS old (READ_ONLY)")
    return con


def normalise(signal: str) -> str:
    """
    Bildet einen alten Kanalnamen auf die neue Schreibweise ab.
    """
    if signal == "HFProbeCounter":
        return "Cycle"
    if signal.startswith("HFBlockEvent_"):
        return "HF_EVENT|" + signal[len("HFBlockEvent_"):]
    return signal

In [ ]:
con = connect()
print("verbunden:", DB_NEW.resolve(), "+", DB_OLD.resolve())

### Teil A – Schema und Signalinventar → Tabelle 6-1

In [ ]:
def part_a(con) -> None:
    header("A  Schema und Signalinventar")

    sub("Spalten beider Tabellen")
    old_cols = con.execute(f"DESCRIBE old.{TBL_OLD}").df()[["column_name", "column_type"]]
    new_cols = con.execute(f"DESCRIBE main.{TBL_NEW}").df()[["column_name", "column_type"]]
    cols = old_cols.merge(new_cols, on="column_name", how="outer",
                          suffixes=("_old", "_new"), indicator=True)
    cols["status"] = cols["_merge"].map(
        {"both": "in beiden", "left_only": "nur alt", "right_only": "nur neu"})
    show(cols[["column_name", "column_type_old", "column_type_new", "status"]]
         .sort_values(["status", "column_name"]))

    sub("Typunterschiede bei gleichnamigen Spalten")
    diff = cols[(cols["_merge"] == "both")
                & (cols["column_type_old"] != cols["column_type_new"])]
    show(diff[["column_name", "column_type_old", "column_type_new"]])

    sub("Kanalnamen: Zuordnung alt zu neu")
    sig_old = con.execute(f"SELECT Signal, COUNT(*) AS n FROM old.{TBL_OLD} "
                          "WHERE Signal IS NOT NULL GROUP BY 1").df()
    sig_new = con.execute(f"SELECT Signal, COUNT(*) AS n FROM main.{TBL_NEW} "
                          "WHERE Signal IS NOT NULL GROUP BY 1").df()
    sig_old["key"] = sig_old["Signal"].map(normalise)
    sig_new["key"] = sig_new["Signal"].map(normalise)
    m = sig_old.merge(sig_new, on="key", how="outer",
                      suffixes=("_old", "_new"), indicator=True)
    renamed = m[(m["_merge"] == "both") & (m["Signal_old"] != m["Signal_new"])]
    only_old = m[m["_merge"] == "left_only"]
    only_new = m[m["_merge"] == "right_only"]

    print(f"gemeinsam       : {(m['_merge'] == 'both').sum()}")
    print(f"davon umbenannt : {len(renamed)}")
    print(f"nur alt         : {len(only_old)}")
    print(f"nur neu         : {len(only_new)}")

    sub("Umbenannt (-> Tabelle 6-1)")
    show(renamed[["Signal_old", "Signal_new"]].sort_values("Signal_old"))
    sub("Nur in der alten Datenbank (weggefallen oder anders zerlegt)")
    show(only_old[["Signal_old", "n_old"]].sort_values("Signal_old"))
    sub("Nur in der neuen Datenbank (neu hinzugekommen)")
    show(only_new[["Signal_new", "n_new"]].sort_values("Signal_new"))

    sub("Blockereignisse: Spalten der alten gegen Kanaele der neuen Datenbank")
    ev_old = sorted(c for c in old_cols["column_name"] if c.startswith("HFBlockEvent_"))
    ev_new = sorted(s for s in sig_new["Signal"] if s.startswith("HF_EVENT|"))
    rows = []
    for c in ev_old:
        field = c[len("HFBlockEvent_"):].lower()
        match = next((s for s in ev_new if s.split("|", 1)[1].lower() == field), None)
        rows.append({"alt_spalte": c, "neu_kanal": match or "FEHLT"})
    for s in ev_new:
        field = s.split("|", 1)[1].lower()
        if not any(c[len("HFBlockEvent_"):].lower() == field for c in ev_old):
            rows.append({"alt_spalte": "neu hinzugekommen", "neu_kanal": s})
    show(pd.DataFrame(rows))

    sub("Mehrfach belegte Kanalnamen in der alten Datenbank")
    print("Ein Kanalname, dessen Zeilenzahl ein Vielfaches der uebrigen betraegt,\n"
          "deutet darauf hin, dass mehrere Signale unter einem Namen zusammengefallen sind.")
    show(con.execute(f"""
        WITH c AS (
            SELECT Signal, COUNT(*) AS n FROM old.{TBL_OLD}
            WHERE DataOrigin = 'HF_Data' GROUP BY 1
        ), md AS (SELECT MEDIAN(n) AS m FROM c)
        SELECT Signal, n, ROUND(n / m, 2) AS vielfaches_des_medians
        FROM c, md WHERE n > 1.5 * m ORDER BY n DESC
    """).df())
    print("Gegenprobe: in check_raw.py Teil M zaehlen, wie oft der Name in der\n"
          "HF-Signalliste vorkommt. Stimmt die Zahl mit dem Vielfachen ueberein,\n"
          "war die Zuordnung ueber das Feld 'Name' nicht eindeutig.")

    for col in ("DataOrigin", "Signal"):
        sub(f"Anteile je {col}: alt gegen neu (groesste Abweichungen zuerst)")
        df = con.execute(f"""
            WITH l AS (SELECT {col} AS k, COUNT(*) AS n FROM old.{TBL_OLD} GROUP BY 1),
                 n AS (SELECT {col} AS k, COUNT(*) AS n FROM main.{TBL_NEW} GROUP BY 1)
            SELECT COALESCE(l.k, n.k) AS {col},
                   COALESCE(l.n, 0) AS alt_zeilen,
                   ROUND(100.0 * COALESCE(l.n, 0) / (SELECT SUM(n) FROM l), 2) AS alt_pct,
                   COALESCE(n.n, 0) AS neu_zeilen,
                   ROUND(100.0 * COALESCE(n.n, 0) / (SELECT SUM(n) FROM n), 2) AS neu_pct
            FROM l FULL OUTER JOIN n ON l.k = n.k
        """).df()
        df["pct_diff"] = (df["neu_pct"] - df["alt_pct"]).abs()
        show(df.sort_values("pct_diff", ascending=False).head(25))

In [ ]:
part_a(con)

### Teil B – Integrität der neuen Datenbank

In [ ]:
def part_b(con) -> None:
    header("B  Integritaet der neuen Datenbank")

    sub("Umfang")
    show(con.execute(f"""
        SELECT COUNT(*) AS zeilen,
               COUNT(DISTINCT Platte) AS platten,
               COUNT(DISTINCT Nut) AS nuten,
               COUNT(DISTINCT Signal) AS kanaele,
               MIN(Time) AS von, MAX(Time) AS bis
        FROM main.{TBL_NEW}
    """).df())

    sub("Duplikate: (Platte, Nut, Signal, Time) mehrfach belegt")
    dup = con.execute(f"""
        SELECT COUNT(*) AS betroffene_schluessel,
               COALESCE(SUM(n) - COUNT(*), 0) AS ueberzaehlige_zeilen
        FROM (SELECT Platte, Nut, Signal, Time, COUNT(*) AS n
              FROM main.{TBL_NEW} GROUP BY 1,2,3,4 HAVING COUNT(*) > 1)
    """).df()
    show(dup)
    print("-> Anforderung erfuellt" if dup.iloc[0, 0] == 0
          else "-> FEHLGESCHLAGEN, Details in Teil C und D")

    sub("Vollstaendigkeit: Kanalzahl je Nut (Abweichler vom Median)")
    show(con.execute(f"""
        WITH per_slot AS (
            SELECT Platte, Nut, COUNT(DISTINCT Signal) AS kanaele, COUNT(*) AS zeilen
            FROM main.{TBL_NEW} GROUP BY 1,2
        ), md AS (SELECT MEDIAN(kanaele) AS median_kanaele FROM per_slot)
        SELECT * FROM per_slot, md WHERE kanaele <> median_kanaele
        ORDER BY Platte, Nut
    """).df(), "keine Abweichler")

    sub(f"Zeitraster HF ({PROBE_SIGNAL}): Abstaende gegen erwartete {HF_PERIOD_MS} ms")
    show(con.execute(f"""
        WITH d AS (
            SELECT date_diff('microsecond', LAG(Time) OVER w, Time) / 1000.0 AS dt_ms
            FROM main.{TBL_NEW}
            WHERE DataOrigin = 'HF_Data' AND Signal = '{PROBE_SIGNAL}'
            WINDOW w AS (PARTITION BY Platte, Nut ORDER BY Time)
        )
        SELECT MEDIAN(dt_ms) AS median_ms, MIN(dt_ms) AS min_ms, MAX(dt_ms) AS max_ms,
               COUNT(*) AS abstaende,
               COUNT(*) FILTER (WHERE ABS(dt_ms - {HF_PERIOD_MS}) > 0.001) AS abweichend
        FROM d WHERE dt_ms IS NOT NULL
    """).df())

    sub("Zyklenzaehler: Luecken je Nut")
    show(con.execute(f"""
        WITH c AS (
            SELECT Platte, Nut, Cycle,
                   Cycle - LAG(Cycle) OVER (PARTITION BY Platte, Nut ORDER BY Cycle) AS d
            FROM (SELECT DISTINCT Platte, Nut, Cycle FROM main.{TBL_NEW}
                  WHERE DataOrigin = 'HF_Data' AND Cycle IS NOT NULL)
        )
        SELECT Platte, Nut, COUNT(*) FILTER (WHERE d > 1) AS luecken,
               COALESCE(SUM(d - 1) FILTER (WHERE d > 1), 0) AS fehlende_zyklen,
               MAX(d) AS groesste_luecke
        FROM c GROUP BY 1,2 HAVING COUNT(*) FILTER (WHERE d > 1) > 0
        ORDER BY fehlende_zyklen DESC
    """).df(), "keine Luecken")

    sub("HF/LF-Synchronisation: Restversatz nach der Korrektur")
    show(con.execute(f"""
        WITH sync AS (
            SELECT Platte, Nut, CAST(ROUND(Value) AS BIGINT) AS Cycle, MIN(Time) AS t_lf
            FROM main.{TBL_NEW}
            WHERE Signal = 'SYNC_CYCLE_HF' AND Value IS NOT NULL
            GROUP BY 1,2,3
        ), hf AS (
            SELECT Platte, Nut, Cycle, MIN(Time) AS t_hf
            FROM main.{TBL_NEW}
            WHERE DataOrigin = 'HF_Data' AND Cycle IS NOT NULL
            GROUP BY 1,2,3
        )
        SELECT h.Platte, h.Nut, COUNT(*) AS stuetzstellen,
               ROUND(MEDIAN(date_diff('microsecond', h.t_hf, s.t_lf)) / 1000.0, 3) AS rest_ms,
               ROUND(STDDEV(date_diff('microsecond', h.t_hf, s.t_lf)) / 1000.0, 3) AS sigma_ms
        FROM hf h JOIN sync s USING (Platte, Nut, Cycle)
        GROUP BY 1,2 ORDER BY ABS(rest_ms) DESC LIMIT 15
    """).df(), "keine gemeinsamen Zyklen")
    print("Erwartung: rest_ms nahe 0. Grosse Werte -> Korrektur greift dort nicht.")

    sub("Stroeme: Zyklus und Position je DataOrigin")
    show(con.execute(f"""
        SELECT DataOrigin, COUNT(*) AS zeilen,
               COUNT(*) FILTER (WHERE Cycle IS NULL) AS ohne_zyklus,
               COUNT(*) FILTER (WHERE WCS_Y_mm IS NULL) AS ohne_position
        FROM main.{TBL_NEW} GROUP BY 1 ORDER BY 2 DESC
    """).df())
    print("Der externe Strom hat eine eigene Zeitbasis (dokumentierte Einschraenkung).")

    sub("Metadaten: funktionale Abhaengigkeit vom Kanalnamen")
    viol = con.execute(f"""
        SELECT Signal,
               COUNT(DISTINCT Unit) AS units,
               COUNT(DISTINCT Axis) AS axes,
               COUNT(DISTINCT SamplingPeriod) AS periods,
               COUNT(DISTINCT Description) AS descriptions
        FROM main.{TBL_NEW} GROUP BY 1
        HAVING units > 1 OR axes > 1 OR periods > 1 OR descriptions > 1
    """).df()
    show(viol, "Pass: jeder Kanal hat genau eine Einheit, Achse, Abtastperiode und Beschreibung.")

    sub("Fehlende Werte je Spalte (Anteil in Prozent)")
    cols = con.execute(f"DESCRIBE main.{TBL_NEW}").df()["column_name"].tolist()
    expr = ", ".join(
        f'ROUND(100.0 * COUNT(*) FILTER (WHERE "{c}" IS NULL) / COUNT(*), 2) AS "{c}"'
        for c in cols)
    nulls = con.execute(f"SELECT {expr} FROM main.{TBL_NEW}").df().T
    nulls.columns = ["null_pct"]
    nulls = nulls[nulls["null_pct"] > 0].sort_values("null_pct", ascending=False)
    print(nulls.to_string() if not nulls.empty else "keine fehlenden Werte")

In [ ]:
part_b(con)

### Teil C – Duplikate

In [ ]:
def part_c(con) -> None:
    header("C  Duplikate")

    sub("Verteilung der Duplikate auf die Stroeme")
    show(con.execute(f"""
        WITH d AS (
            SELECT DataOrigin, Platte, Nut, Signal, Time, COUNT(*) AS n
            FROM main.{TBL_NEW} GROUP BY 1,2,3,4,5
        )
        SELECT DataOrigin,
               COUNT(*) FILTER (WHERE n > 1) AS mehrfach_belegte_zeitpunkte,
               COALESCE(SUM(n - 1) FILTER (WHERE n > 1), 0) AS ueberzaehlige_zeilen,
               MAX(n) AS groesste_gruppe
        FROM d GROUP BY 1 ORDER BY 3 DESC
    """).df())

    sub("Gleiche oder verschiedene Werte innerhalb einer Gruppe")
    show(con.execute(f"""
        WITH d AS (
            SELECT DataOrigin, Platte, Nut, Signal, Time,
                   COUNT(*) AS n, COUNT(DISTINCT Value) AS verschiedene_werte
            FROM main.{TBL_NEW} GROUP BY 1,2,3,4,5 HAVING COUNT(*) > 1
        )
        SELECT DataOrigin, COUNT(*) AS gruppen,
               COUNT(*) FILTER (WHERE verschiedene_werte = 1) AS echte_wiederholung,
               COUNT(*) FILTER (WHERE verschiedene_werte > 1) AS verschiedene_messwerte
        FROM d GROUP BY 1
    """).df(), "keine Duplikate")
    print("verschiedene_messwerte > 0 -> eigenstaendige Samples, die sich nur einen\n"
          "Zeitstempel teilen: die Zeitaufloesung ist das Problem (Werte mit gleichem Zeitstempel, auch bei us speicherung) \n"
          ")

    sub("Welche Signale tragen die Duplikate")
    show(con.execute(f"""
        SELECT Signal, DataOrigin, COUNT(*) AS dup_gruppen,
               SUM(n - 1) AS ueberzaehlige_zeilen,
               MAX(verschiedene_werte) AS max_verschiedene_werte
        FROM (SELECT Signal, DataOrigin, Platte, Nut, Time,
                     COUNT(*) AS n, COUNT(DISTINCT Value) AS verschiedene_werte
              FROM main.{TBL_NEW} GROUP BY 1,2,3,4,5 HAVING COUNT(*) > 1)
        GROUP BY 1,2 ORDER BY ueberzaehlige_zeilen DESC LIMIT 20
    """).df(), "keine Duplikate")

    sub("Tatsaechliche Aufloesung der Zeitstempel je Strom")
    show(con.execute(f"""
        WITH t AS (SELECT DISTINCT DataOrigin, Time FROM main.{TBL_NEW}),
        d AS (
            SELECT DataOrigin, date_diff('microsecond', LAG(Time) OVER w, Time) AS dt_us
            FROM t WINDOW w AS (PARTITION BY DataOrigin ORDER BY Time)
        )
        SELECT DataOrigin, MIN(dt_us) AS min_us, MEDIAN(dt_us) AS median_us,
               COUNT(*) FILTER (WHERE dt_us % 1000 = 0) AS auf_ms_gerundet,
               COUNT(*) AS abstaende
        FROM d WHERE dt_us IS NOT NULL AND dt_us > 0
        GROUP BY 1 ORDER BY 1
    """).df())
    print("auf_ms_gerundet ~ abstaende -> alle Zeitstempel liegen auf vollen\n"
          "Millisekunden, die Umrechnung hat die feinere Aufloesung verworfen.")

    sub("Abtastrate des externen Stroms (ET200), aus dem Bestand geschaetzt")
    show(con.execute(f"""
        SELECT Signal, Platte, Nut, COUNT(*) AS samples,
               ROUND(date_diff('microsecond', MIN(Time), MAX(Time)) / 1e6, 3) AS dauer_s,
               ROUND(COUNT(*) / NULLIF(date_diff('microsecond', MIN(Time), MAX(Time)) / 1e6, 0), 1)
                   AS effektiv_hz
        FROM main.{TBL_NEW} WHERE DataOrigin = 'ET200_Data'
        GROUP BY 1,2,3 ORDER BY Nut, Signal LIMIT 9
    """).df(), "kein ET200_Data-Strom")

In [ ]:
part_c(con)

### Teil D – Metadaten-Fanout

In [ ]:
def part_d(con) -> None:
    header("D  Metadaten-Fanout: verdoppelt ein doppelter Eintrag die Zeilen?")

    sub(f"{SUSPECT_SIGNAL}: Zeilen je Einheit und Beschreibung")
    show(con.execute(f"""
        SELECT Signal, Unit, Description, COUNT(*) AS zeilen
        FROM main.{TBL_NEW} WHERE Signal = '{SUSPECT_SIGNAL}'
        GROUP BY 1,2,3 ORDER BY zeilen DESC
    """).df(), f"{SUSPECT_SIGNAL} nicht vorhanden")
    print("Zwei Zeilen mit gleicher Anzahl -> jede Messzeile wurde verdoppelt.")

    sub("Alle Signale mit mehr als einer Einheit oder Beschreibung")
    show(con.execute(f"""
        SELECT Signal, COUNT(DISTINCT Unit) AS units,
               COUNT(DISTINCT Description) AS descriptions, COUNT(*) AS zeilen
        FROM main.{TBL_NEW} GROUP BY 1
        HAVING units > 1 OR descriptions > 1 ORDER BY zeilen DESC
    """).df())
    print("Gegenprobe der Signallisten: check_raw.py Teil M.")

In [ ]:
part_d(con)

### Teil E – WCS_Y_mm

In [ ]:
def part_e(con, platte, nut: int) -> None:
    header("E  WCS_Y_mm: Belegung und Verlauf")

    sub("Belegung je Strom")
    show(con.execute(f"""
        SELECT DataOrigin, COUNT(*) AS zeilen, COUNT(WCS_Y_mm) AS mit_wert,
               ROUND(100.0 * COUNT(WCS_Y_mm) / COUNT(*), 2) AS pct_belegt,
               ROUND(MIN(WCS_Y_mm), 3) AS min, ROUND(MAX(WCS_Y_mm), 3) AS max,
               ROUND(AVG(WCS_Y_mm), 3) AS mittel
        FROM main.{TBL_NEW} GROUP BY 1 ORDER BY 2 DESC
    """).df())

    sub("Wertebereich je Nut (Auszug)")
    show(con.execute(f"""
        SELECT Platte, Nut, COUNT(WCS_Y_mm) AS mit_wert,
               ROUND(MIN(WCS_Y_mm), 3) AS min, ROUND(MAX(WCS_Y_mm), 3) AS max,
               ROUND(MAX(WCS_Y_mm) - MIN(WCS_Y_mm), 3) AS spanne
        FROM main.{TBL_NEW} WHERE WCS_Y_mm IS NOT NULL
        GROUP BY 1,2 ORDER BY Platte, Nut LIMIT 10
    """).df())

    sub(f"Verlauf ueber Nut {nut} der Platte {platte}, 20 Stuetzstellen")
    show(con.execute(f"""
        WITH d AS (
            SELECT Duration_Seconds AS t, WCS_Y_mm AS y,
                   NTILE(20) OVER (ORDER BY Duration_Seconds) AS bucket
            FROM main.{TBL_NEW}
            WHERE Platte = {lit(platte)} AND Nut = {nut} AND WCS_Y_mm IS NOT NULL
        )
        SELECT bucket, ROUND(MIN(t), 3) AS t_von, ROUND(MAX(t), 3) AS t_bis,
               ROUND(MIN(y), 3) AS y_min, ROUND(MAX(y), 3) AS y_max
        FROM d GROUP BY bucket ORDER BY bucket
    """).df(), "keine Werte fuer diese Nut")

    sub("Verteilung der Werte, Quantile")
    show(con.execute(f"""
        SELECT ROUND(MIN(WCS_Y_mm), 3) AS p0,
               ROUND(QUANTILE_CONT(WCS_Y_mm, 0.05), 3) AS p5,
               ROUND(QUANTILE_CONT(WCS_Y_mm, 0.25), 3) AS p25,
               ROUND(MEDIAN(WCS_Y_mm), 3) AS p50,
               ROUND(QUANTILE_CONT(WCS_Y_mm, 0.75), 3) AS p75,
               ROUND(QUANTILE_CONT(WCS_Y_mm, 0.95), 3) AS p95,
               ROUND(MAX(WCS_Y_mm), 3) AS p100
        FROM main.{TBL_NEW} WHERE WCS_Y_mm IS NOT NULL
    """).df())

    sub("Zum Vergleich: Rohwert des Geberkanals ENC_POS|2, aus dem WCS abgeleitet wird")
    show(con.execute(f"""
        SELECT ROUND(MIN(Value), 3) AS min, ROUND(MEDIAN(Value), 3) AS median,
               ROUND(MAX(Value), 3) AS max, COUNT(*) AS zeilen
        FROM main.{TBL_NEW} WHERE Signal = 'ENC_POS|2' AND Value IS NOT NULL
    """).df())

In [ ]:
part_e(con, PLATTE, NUT)

### Teil F – Zugriffsleistung

In [ ]:
def _walk(node: dict, out: list) -> None:
    """Sammelt Name und Kardinalitaet aller Knoten eines DuckDB-Profils."""
    name = node.get("operator_name") or node.get("name") or node.get("operator_type") or ""
    card = node.get("operator_cardinality")
    if card is None:
        card = node.get("cardinality")
    if name:
        out.append((str(name), card))
    for child in node.get("children") or []:
        _walk(child, out)


def scanned_rows(con, query: str) -> int | None:
    """Fuehrt die Abfrage mit JSON-Profiling aus und liest die groesste
    Kardinalitaet eines Scan-Knotens - belastbarer als der EXPLAIN-Text."""
    path = Path(tempfile.gettempdir()) / f"duckprof_{os.getpid()}.json"
    con.execute("PRAGMA enable_profiling='json'")
    con.execute(f"PRAGMA profiling_output='{path.as_posix()}'")
    try:
        con.execute(query).fetchall()
    finally:
        con.execute("PRAGMA disable_profiling")
    nodes: list = []
    try:
        with open(path, encoding="utf-8") as f:
            _walk(json.load(f), nodes)
    except Exception as exc:
        print(f"  (Profil nicht lesbar: {exc})")
        return None
    finally:
        path.unlink(missing_ok=True)
    scans = [c for n, c in nodes if "SCAN" in n.upper() and isinstance(c, (int, float))]
    return int(max(scans)) if scans else None


def timed(con, query: str, repeats: int = 5) -> dict:
    con.execute(query).fetchall()
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        rows = con.execute(query).fetchall()
        runs.append(time.perf_counter() - t0)
    return {"median_s": round(sorted(runs)[len(runs) // 2], 4),
            "min_s": round(min(runs), 4),
            "ergebniszeilen": len(rows),
            "scan_zeilen": scanned_rows(con, query)}


def part_f(con) -> pd.DataFrame:
    header("F  Zugriffsleistung")

    plate = con.execute(f"SELECT MIN(Platte) FROM main.{TBL_NEW}").fetchone()[0]
    slot = con.execute(f"SELECT MIN(Nut) FROM main.{TBL_NEW} "
                       f"WHERE Platte = {lit(plate)}").fetchone()[0]
    total = con.execute(f"SELECT COUNT(*) FROM main.{TBL_NEW}").fetchone()[0]

    queries = {
        "Q1 Ausschnitt einer Nut, wenige Kanaele": f"""
            SELECT Time, Signal, Value FROM main.{TBL_NEW}
            WHERE Platte = {lit(plate)} AND Nut = {slot}
              AND Signal IN ('{PROBE_SIGNAL}', 'CURRENT|3', 'CTRL_DIFF|3')
              AND Duration_Seconds BETWEEN 1 AND 3""",
        "Q2 ein Kanal ueber alle Nuten einer Platte": f"""
            SELECT Nut, Time, Value FROM main.{TBL_NEW}
            WHERE Platte = {lit(plate)} AND Signal = '{PROBE_SIGNAL}'""",
        "Q3 Aggregat ueber die Kampagne": f"""
            SELECT Platte, Nut, AVG(Value) AS mittel, STDDEV(Value) AS streuung, COUNT(*) AS n
            FROM main.{TBL_NEW} WHERE Signal = '{PROBE_SIGNAL}'
            GROUP BY 1,2 ORDER BY 1,2""",
        "Q4 Kanalverzeichnis ohne Messwerte": f"""
            SELECT DISTINCT Signal, Unit, Description FROM main.{TBL_NEW}""",
    }

    rows = []
    for name, q in queries.items():
        r = timed(con, q)
        scan = r["scan_zeilen"]
        r["anteil_tabelle_pct"] = round(100.0 * scan / total, 1) if scan else None
        r["abfrage"] = name
        rows.append(r)
        anteil = f"{r['anteil_tabelle_pct']:5.1f} %" if scan else "  n/a"
        print(f"{name:45s} {r['median_s']:8.4f} s  Ergebnis {r['ergebniszeilen']:>9,}  "
              f"Scan {str(scan):>12}  {anteil}")

    print(f"\nTabelle gesamt: {total:,} Zeilen")
    print("Der Anteil ist die Groesse, auf die es ankommt. Liegt er bei 100 Prozent,\n"
          "wird trotz der Bedingung die ganze Tabelle gelesen und das Pruning greift nicht.")

    sub("Speicherbedarf")
    size_new = DB_NEW.stat().st_size / 1e6
    size_old = DB_OLD.stat().st_size / 1e6
    print(f"{DB_NEW.name}: {size_new:8.1f} MB bei {total:,} Zeilen "
          f"({size_new * 1e6 / max(total, 1):.1f} Byte je Zeile)")
    print(f"{DB_OLD.name}: {size_old:8.1f} MB")
    print("Hinweis: verschiedene Kampagnen, daher nur Byte je Zeile vergleichbar.")

    df = pd.DataFrame(rows)[["abfrage", "median_s", "min_s", "ergebniszeilen",
                             "scan_zeilen", "anteil_tabelle_pct"]]
    df.to_csv(OUT_CSV, index=False)
    print(f"\n-> {OUT_CSV} geschrieben")
    return df

In [ ]:
messung = part_f(con)
messung

In [ ]:
con.close()